# Synthetic Monitoring Demo

This notebook demonstrates a simple governance-oriented workflow for monitoring a synthetic small business underwriting population. It is a demonstration artifact only and does not use production data.


## Objectives

- create a synthetic applicant population
- simulate underwriting decisions and explanations
- review simple performance and drift signals
- review disparity indicators for governance escalation
- show how monitoring results could be summarized


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 500

df = pd.DataFrame(
    {
        "segment": rng.choice(["segment_a", "segment_b"], size=n, p=[0.6, 0.4]),
        "revenue_band": rng.choice(["low", "mid", "high"], size=n, p=[0.35, 0.45, 0.20]),
        "utilization": rng.normal(0.55, 0.18, size=n).clip(0, 1),
        "delinq_flag": rng.choice([0, 1], size=n, p=[0.82, 0.18]),
        "cashflow_index": rng.normal(0.0, 1.0, size=n),
    }
)

df.head()


In [ ]:
score = (
    0.9 * df["cashflow_index"]
    - 1.2 * df["utilization"]
    - 1.1 * df["delinq_flag"]
    + np.where(df["revenue_band"] == "high", 0.35, 0)
)

df["score"] = score
df["approved"] = (df["score"] > -0.35).astype(int)
df["reason_code"] = np.select(
    [
        df["delinq_flag"] == 1,
        df["utilization"] > 0.75,
        df["cashflow_index"] < -0.5,
    ],
    [
        "recent_delinquency",
        "high_utilization",
        "weak_cashflow_signal",
    ],
    default="meets_threshold",
)

df[["score", "approved", "reason_code"]].head()


In [ ]:
approval_rates = df.groupby("segment")["approved"].mean().rename("approval_rate")
reason_mix = (
    df.groupby(["segment", "reason_code"]).size().rename("count").reset_index()
)

approval_rates, reason_mix.head(10)


## Next Extensions

- add baseline versus current-period drift comparisons
- add simple threshold logic for escalation
- add explanation stability checks across periods
- add a governance summary table for reporting
